In [ ]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
import spacy
from textblob import Word
from transformers import BertTokenizer, BertModel
import torch

# Download NLTK data files (run this once)
nltk.download('punkt')
nltk.download('wordnet')

# Initialize NLTK tools
lemmatizer = WordNetLemmatizer()

# Initialize SpaCy
nlp = spacy.load('en_core_web_sm')

# Initialize BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def expand_contractions(text):
    contractions = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hasn't": "has not", "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "I'd": "I would", "I'll": "I will", "I'm": "I am", "I've": "I have",
    "isn't": "is not", "it'd": "it would", "it'll": "it will", "it's": "it is",
    "let's": "let us", "mightn't": "might not", "mustn't": "must not", "shan't": "shall not",
    "she'd": "she would", "she'll": "she will", "she's": "she is", "shouldn't": "should not",
    "that's": "that is", "there's": "there is", "they'd": "they would", "they'll": "they will",
    "they're": "they are", "they've": "they have", "we'd": "we would", "we'll": "we will",
    "we're": "we are", "we've": "we have", "weren't": "were not", "what'll": "what will",
    "what're": "what are", "what's": "what is", "what've": "what have", "where's": "where is",
    "who'd": "who would", "who'll": "who will", "who's": "who is", "won't": "will not",
    "wouldn't": "would not", "you'd": "you would", "you'll": "you will", "you're": "you are",
    "you've": "you have", "n't": " not", "'re": " are", "'s": " is", "'d": " would",
    "'ll": " will", "'t": " not", "'ve": " have", "'m": " am", "'ne": " not"
    }


def normalize_text(text):
    # Expand contractions dynamically
    text = expand_contractions(text)

    # Normalize numeric values
    text = re.sub(r'\b\d+\b', 'NUMBER', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def preprocess_chunk(chunk):
    # Normalize text
    chunk = normalize_text(chunk)

    # Convert to lowercase
    chunk = chunk.lower()

    # Remove punctuation and special characters
    chunk = re.sub(r'[^\w\s]', '', chunk)

    # Tokenize the chunk
    tokens = nltk.word_tokenize(chunk)

    # Lemmatize tokens
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Rejoin tokens into a single string
    cleaned_chunk = ' '.join(tokens)

    return cleaned_chunk

def get_bert_embedding(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**tokens)
    # Use the mean of the last hidden state
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze()
    return embedding


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
  # Example usage
chunk = "As per the Allocation of Business (Rules), 1961, Department of Justice is \
a part of Ministry of Law & Justice, Government of India. It is one of the oldest Ministries \
of the Government of India. Till 31.12.2009, Department of Justice was part of Ministry of Home \
Affairs and Union Home Secretary had been the Secretary of Department of Justice. Keeping in view \
the increasing workload and formulating many policies and programmes on Judicial Reforms in the \
country, a separate Department namely Department of Justice was carved out from MHA and placed under the \
charge of Secretary to Government of India and it started working as such from 1st January, 2010 under \
the Ministry of Law & Justice. The Department is housed in the Jaisalmer House, 26, Man Singh Road, New Delhi. \
The Organizational setup of the Department includes 04 Joint Secretaries, 08 Directors/ Deputy Secretaries and 09 Under Secretaries. \
The functions of the Department of Justice include the appointment, resignation and removal of the Chief Justice of India, Judges of \
the Supreme Court of India, Chief Justices and Judges of the High Courts and their service matters. In addition, the Department implements \
important schemes for Development of Infrastructure Facilities for Judiciary, setting up of Special Courts for speedy trial and disposal of \
cases of sensitive nature (Fast Track Special Court for cases of rape and POCSO Act), E-court Project on computerization of \
various courts across the country, legal aid to poor and access to justice, financial assistance to National Judicial Academy \
for providing training to the Judicial Officers of the country. The functions of Department of Justice are given in Allocation of Business (Rules), 1961."
cleaned_chunk = preprocess_chunk(chunk)
embedding = get_bert_embedding(cleaned_chunk)

KeyboardInterrupt: 